In [1]:
import os
import json
import pandas as pd
import numpy as np
from report_fct import filter_interval
import pandas as pd

def build_csv_from_summary(summary_path, data_root, output_csv="all_data.csv"):
    # Load the summary JSON
    with open(summary_path, "r") as f:
        summary = json.load(f)

    all_rows = []

    # Iterate over each run entry in the summary
    for run_entry in summary:
        run_name = run_entry["run"]
        date = run_entry["date"]  
        person = run_entry["person"]
        intervals = run_entry["intervals"]

        # Ensure intervals is a list
        if not isinstance(intervals, list):
            print(f"⚠️ Invalid interval data for {run_name}: expected a list, found {type(intervals)}")
            continue

        # Build the run folder path using date, person, and run name
        run_path = os.path.join(data_root, date, person, run_name)
        # Check if the run folder exists
        if not os.path.isdir(run_path):
            print(f"⚠️ Run folder not found for {run_name} at {run_path}")
            continue

        # Search for the CSV file in the run folder
        csv_files = [f for f in os.listdir(run_path) if f.endswith(".csv")]
        if len(csv_files) != 1:
            print(f"⚠️ Skipping {run_name}: expected 1 CSV, found {len(csv_files)}")
            continue

        # Build the full path to the CSV file
        csv_path = os.path.join(run_path, csv_files[0])

        # Read the CSV file into a DataFrame
        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"⚠️ Error reading CSV file {csv_path}: {e}")
            continue

        # Process each interval in the run
        try:
            print(f"✔ Processing run: {run_name}, total intervals: {len(intervals)}")

            # for i, interval in enumerate(intervals):
            #     start, end = interval["start_time"], interval["end_time"]
            #     df_filtered = filter_interval(df, start, end)
            #     # Add necessary metadata to each row
            #     df_filtered["run"] = run_name
            #     df_filtered["rider_name"] = person
            #     df_filtered["boat_name"] = csv_files[0].replace(".csv", "")
            #     df_filtered["maneuver_index"] = interval["maneuver_index"]
            #     df_filtered["maneuver_type"] = interval["maneuver_type"]
            #     df_filtered["interval_duration"] = interval["duration"]
            #     df_filtered["start_time"] = interval["start_time"]
            #     df_filtered["end_time"] = interval["end_time"]
         
            #     lines = df_filtered[["lateral", "central"]].values
            #     sorted_lines = np.sort(lines, axis=1)
                
            #     all_rows.append(df_filtered)

            # Process each interval in the run
            for i, interval in enumerate(intervals):
                            m_index = interval["maneuver_index"]
                            start_m = interval["start_time"]
                            end_m = interval["end_time"]
                            
                            # --- GESTION DE LA SUPERPOSITION ---
                            reference_start = start_m - 10
                            
                            # Si ce n'est pas la première manœuvre, on vérifie la précédente
                            if i > 0:
                                prev_end = intervals[i-1]["maneuver_time"]+2
                                if reference_start < prev_end:
                                    print(f"⚠️ Overlap detected in {run_name} between maneuver {i} and {i-1}.")
                                    raise ValueError(f"Overlap detected in {run_name} between maneuver {i} and {i-1}.")
                            df_interval_complet = filter_interval(df, reference_start, end_m)
                            df_filtered = df_interval_complet.copy()

                            # 3. Métadonnées
                            df_filtered["run"] = run_name
                            df_filtered["rider_name"] = person
                            df_filtered["boat_name"] = csv_files[0].replace(".csv", "")
                            df_filtered["maneuver_type"] = interval["maneuver_type"]
                            df_filtered["interval_duration"] = interval["duration"]
                            df_filtered["start_time"] = start_m
                            df_filtered["end_time"] = end_m

                            # 4. Marquage 0 vs Index
                            is_inside_maneuver = (df_filtered['SecondsSince1970'] >= start_m) & \
                                                (df_filtered['SecondsSince1970'] <= end_m)
                            
                            df_filtered["maneuver_index"] = np.where(is_inside_maneuver, m_index, 0)
                            df_filtered["target_id"] = m_index

                            all_rows.append(df_filtered)

        except Exception as e:
            print(f"❌ Error processing run {run_name}, interval {i + 1}: {e}")
            continue

    # Final save
    if not all_rows:
        print("❌ No valid data found.")
        return

    # Combine all rows into a single DataFrame and save to CSV
    df_global = pd.concat(all_rows, ignore_index=True)
    df_global = df_global.sort_values(by='SecondsSince1970', ascending=True)
    df_global.to_csv(output_csv, index=False)
    print(f"✅ Global CSV saved to: {output_csv}")

In [2]:
build_csv_from_summary(
    summary_path="summary.json",
    # data_root="../Data_Sailnjord/Maneuvers",
    data_root = "../Data_Sailnjord/Hyères November 2025/Maneuvers",
    output_csv="all_data.csv"
)

✔ Processing run: 30_11_2025_Run11, total intervals: 10
✔ Processing run: 30_11_2025_Run12, total intervals: 5
✅ Global CSV saved to: all_data.csv
